# Resolución de los Ejercicios – Estadística Descriptiva y Análisis Univariado

Este notebook contiene las soluciones comentadas de los ejercicios propuestos.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import boxcox

sns.set_theme(style="whitegrid")
np.random.seed(123)

n = 400
df = pd.DataFrame({
    "ingresos": np.random.lognormal(mean=3.2, sigma=0.7, size=n),
    "edad": np.random.normal(loc=38, scale=12, size=n).clip(18, 75),
    "satisfaccion": np.random.randint(1, 6, size=n),
    "region": np.random.choice(["Norte", "Sur", "Este", "Oeste"], size=n, p=[0.35, 0.25, 0.25, 0.15]),
    "compra": np.random.binomial(1, 0.4, size=n)
})
df.loc[np.random.choice(df.index, 8, replace=False), "ingresos"] *= 4
df.head()


## Ejercicio 1 – Solución


In [ ]:
def resumen(s, nombre):
    print(f"=== {nombre} ===")
    print(f"Media     : {s.mean():.3f}")
    print(f"Mediana   : {s.median():.3f}")
    print(f"Std       : {s.std():.3f}")
    print(f"IQR       : {s.quantile(0.75) - s.quantile(0.25):.3f}")
    print(f"Sesgo     : {s.skew():.3f}")
    print()

resumen(df["ingresos"], "Ingresos")
resumen(df["edad"], "Edad")

print("La variable ingresos presenta un sesgo positivo mucho mayor que la edad.")
print("Por eso la media de ingresos se aleja más de la mediana (los valores extremos tiran la media hacia arriba).")


## Ejercicio 2 – Solución


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sns.histplot(df["ingresos"], kde=True, ax=axes[0], color="coral")
axes[0].set_title("Ingresos (hist + KDE)")

sns.histplot(df["edad"], kde=True, ax=axes[1], color="steelblue")
axes[1].set_title("Edad (hist + KDE)")

sns.boxplot(y=df["ingresos"], ax=axes[2], color="coral")
axes[2].set_title("Boxplot Ingresos")

plt.tight_layout()
plt.show()

print("Ingresos: distribución claramente sesgada a la derecha, con outliers visibles en el boxplot.")
print("Edad: distribución aproximadamente simétrica, sin outliers evidentes.")


## Ejercicio 3 – Solución


In [ ]:
freq = df["region"].value_counts()
prop = df["region"].value_counts(normalize=True)
tabla = pd.DataFrame({"Frecuencia": freq, "Proporción": prop.round(3)})
print(tabla)

orden = freq.index.tolist()
plt.figure(figsize=(7, 4))
sns.countplot(x="region", data=df, order=orden, palette="viridis")
plt.title("Frecuencia de regiones (ordenada)")
plt.show()

print("Ninguna categoría está por debajo del 5 %. Si hubiera una categoría rara,")
print("la agruparíamos en 'Otros' documentando la decisión, o la excluiríamos si no aporta valor analítico.")


## Ejercicio 4 – Solución


In [ ]:
def outliers_iqr(serie, factor=1.5):
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    return (serie < q1 - factor * iqr) | (serie > q3 + factor * iqr)

def outliers_zscore(serie, umbral=3.0):
    z = (serie - serie.mean()) / serie.std()
    return np.abs(z) > umbral

mask_iqr = outliers_iqr(df["ingresos"])
mask_z = outliers_zscore(df["ingresos"])

print(f"Outliers IQR   : {mask_iqr.sum()}")
print(f"Outliers Z-score: {mask_z.sum()}")

print("\nPara ingresos (distribución sesgada) el método IQR es preferible,")
print("porque no asume normalidad y es robusto al sesgo. El Z-score se ve afectado")
print("por la propia media y desviación influidas por los outliers.")


## Ejercicio 5 – Solución


In [ ]:
df["ingresos_log"] = np.log(df["ingresos"])
df["ingresos_bc"], lam = boxcox(df["ingresos"])

print(f"Lambda Box-Cox: {lam:.3f}")
print(f"Sesgo original : {df['ingresos'].skew():.3f}")
print(f"Sesgo log      : {df['ingresos_log'].skew():.3f}")
print(f"Sesgo Box-Cox  : {df['ingresos_bc'].skew():.3f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(df["ingresos"], kde=True, ax=axes[0], color="coral")
axes[0].set_title("Original")
sns.histplot(df["ingresos_log"], kde=True, ax=axes[1], color="steelblue")
axes[1].set_title("Log")
sns.histplot(df["ingresos_bc"], kde=True, ax=axes[2], color="seagreen")
axes[2].set_title(f"Box-Cox (λ={lam:.2f})")
plt.tight_layout()
plt.show()


## Ejercicio 6 – Solución


In [ ]:
jovenes = df.loc[df["edad"] < 40, "ingresos"]
mayores = df.loc[df["edad"] >= 40, "ingresos"]

t_stat, p_t = stats.ttest_ind(jovenes, mayores, equal_var=False)
u_stat, p_u = stats.mannwhitneyu(jovenes, mayores, alternative="two-sided")

print(f"t-test de Welch : t = {t_stat:.3f}, p = {p_t:.4f}")
print(f"Mann-Whitney U  : U = {u_stat:.1f}, p = {p_u:.4f}")

alpha = 0.05
print(f"\n¿Rechazar H0 (t-test)? {'Sí' if p_t < alpha else 'No'}")
print(f"¿Rechazar H0 (MW)?    {'Sí' if p_u < alpha else 'No'}")


## Ejercicio 7 – Solución


In [ ]:
print("Pearson :", df["edad"].corr(df["ingresos"], method="pearson").round(3))
print("Spearman:", df["edad"].corr(df["ingresos"], method="spearman").round(3))
print("Kendall :", df["edad"].corr(df["ingresos"], method="kendall").round(3))

sns.jointplot(data=df, x="edad", y="ingresos", kind="reg", height=5, 
              joint_kws={"line_kws": {"color": "red"}})
plt.show()

print("Los coeficientes son bajos en valor absoluto → evidencia débil de asociación lineal o monotónica")
print("entre edad e ingresos en este dataset sintético.")


---
**Fin de la resolución.**
